# TFG V6: QFT-based interference for window search

This notebook explores the contiguous free-region search problem on discrete grids using QFT-based interference. The index register is initialized directly in the valid search domain, so the initial state is the uniform superposition over window start positions \(i=0,\ldots,W-1\), and the grid itself remains classical.

The main idea is to encode structural information about each candidate window as a phase profile on the index register, then use the Quantum Fourier Transform to convert that phase structure into amplitude redistribution. This avoids Grover's diffusion operator and instead uses Fourier sensitivity to periodicity, smooth cost changes, and regular geometry in the sliding-window cost signal.

Three phase modes are implemented. `cost_linear` applies a phase proportional to the number of occupied cells in the window. `cost_exponential` increases the penalty nonlinearly with cost. `fourier_encoded` computes the dominant Fourier component of the classical cost profile and uses it to build a phase pattern aligned with the Fourier basis.

Two QFT strategies are compared. In the single-QFT strategy, phase and local mixer layers are applied first and a QFT is used only as the final readout transform. In the sandwich strategy, each layer applies `U_phi -> QFT -> mixer -> iQFT`, which makes the mixer act as a convolution-like global operation in the original position basis while returning to the computational basis before the next phase layer.

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

from math import prod
import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, PhaseGate, UnitaryGate

plt.rcParams.update({
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

COLOR_VALID = "#2ecc71"
COLOR_INVALID = "#e74c3c"
COLOR_BASELINE = "#7f8c8d"

print(qiskit.__version__)


In [ ]:
# =========================================================
# Parametros configurables del experimento principal
# =========================================================

N = [8]
M = [2]
occupied_coords = [(0,), (1,), (2,), (6,), (7,)]

# Parametros QFT
phase_mode = "cost_linear"   # "cost_linear", "cost_exponential", "fourier_encoded"
theta = np.pi / 3
alpha = 2.0                  # solo para cost_exponential
freq_target = None           # None = dominante automatico, int = frecuencia fija

# Parametros de capas
mixer_angle = 0.40
repetitions = 4
qft_sandwich = False         # False: QFT solo al final; True: QFT sandwich por capa
use_qft = True
use_mixer = True
qft_inverse = False          # True: usar iQFT en lugar de QFT
mixer_method = "linear_valid"


In [ ]:
# =========================================================
# Geometria ND y utilidades clasicas
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index.
    Example with dims = [4,4]:
    (0,0) -> 0    (0,1) -> 1    (0,2) -> 2    (0,3) -> 3
    (1,0) -> 4    (1,1) -> 5    (1,2) -> 6    (1,3) -> 7
    (2,0) -> 8    (2,1) -> 9    (2,2) -> 10   (2,3) -> 11
    (3,0) -> 12   (3,1) -> 13   (3,2) -> 14   (3,3) -> 15
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates.
    Example with dims = [4,4]:
    0 -> (0,0)    1 -> (0,1)    2 -> (0,2)    3 -> (0,3)
    4 -> (1,0)    5 -> (1,1)    6 -> (1,2)    7 -> (1,3)
    8 -> (2,0)    9 -> (2,1)    10 -> (2,2)   11 -> (2,3)
    12 -> (3,0)   13 -> (3,1)    14 -> (3,2)   15 -> (3,3)
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def window_string_classical(grid_bits, start, N, M):
    return ''.join(str(grid_bits[q]) for q in window_qubits_nd(start, N, M))


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    """
    000 -> 0
    001 -> 1
    011 -> 3
    010 -> 2
    110 -> 6
    111 -> 7
    101 -> 5
    100 -> 4
    """
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")


In [ ]:
# =========================================================
# Bloques cuanticos: QFT, fase y mixer
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    """Prepares 1/sqrt(W) sum_{i=0}^{W-1} |i>, even when W is not a power of 2."""
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


def append_mcx(qc, controls, target):
    """Modern MCX through MCXGate, without deprecated arguments such as mode='noancilla'."""
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    else:
        qc.append(MCXGate(num_ctrl_qubits=len(controls)), controls + [target])


def apply_window_loader(qc, n, idx, m, starts, N, M, order_valid):
    """
    Implements L: |grid>|i>|m> -> |grid>|i>|m xor window_i>.
    Applying this function twice uncomputes the window because the block is reversible XOR.
    """
    IDX = len(idx)
    current_zero_mask = [False] * IDX

    for i in order_valid:
        bits = [(i >> b) & 1 for b in range(IDX)]  # little-endian en Qiskit
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]

        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]

        for j, n_pos in enumerate(window_qubits_nd(starts[i], N, M)):
            controls = [idx[b] for b in range(IDX)] + [n[n_pos]]
            append_mcx(qc, controls, m[j])

    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def two_level_mixer_gate(num_qubits, a, b, beta, label=None):
    """
    Local rotation in the subspace span{|a>, |b>}:
        exp(-i beta X_ab)
    and identity action on the rest. This is a modular prototype for small graphs.
    """
    dim = 2**num_qubits
    U = np.eye(dim, dtype=complex)
    c = np.cos(beta)
    s = -1j * np.sin(beta)
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return UnitaryGate(U, label=label or f"Mix({a},{b})")


def mixer_edges_from_starts(starts, N, method="local_geometric"):
    """Local edges between valid indices. Does not implement Grover global diffusion."""
    if method == "linear_valid":
        return [(i, i + 1) for i in range(len(starts) - 1)]

    if method != "local_geometric":
        raise ValueError("Unknown local mixer method.")

    start_to_idx = {tuple(s): i for i, s in enumerate(starts)}
    edges = []
    D = len(N)
    for i, start in enumerate(starts):
        for d in range(D):
            neigh = list(start)
            neigh[d] += 1
            neigh = tuple(neigh)
            if neigh in start_to_idx:
                edges.append((i, start_to_idx[neigh]))
    return edges


def apply_phase_to_basis_state(qc, idx, basis_state, angle):
    """Applies exp(i angle) to the computational state |basis_state> of the idx register."""
    if abs(angle) < 1e-15:
        return

    IDX = len(idx)
    zero_qubits = [q for bit, q in enumerate(idx) if ((basis_state >> bit) & 1) == 0]
    for q in zero_qubits:
        qc.x(q)

    if IDX == 1:
        qc.p(angle, idx[0])
    else:
        gate = PhaseGate(angle).control(IDX - 1)
        qc.append(gate, list(idx[:-1]) + [idx[-1]])

    for q in reversed(zero_qubits):
        qc.x(q)


def apply_qft(qc, register, inverse=False, do_swaps=True):
    """
    Applies QFT or inverse QFT to `register` (a QuantumRegister or list 
    of qubits). Standard textbook QFT circuit: Hadamard + controlled 
    phase rotations R_k = [[1,0],[0,exp(2*pi*i/2^k)]]. Bit-reversal swaps 
    at the end if do_swaps=True.
    If inverse=True, applies QFT^dagger (reverse order, negate angles).
    """
    qubits = list(register)
    n = len(qubits)

    if inverse:
        if do_swaps:
            for j in range(n // 2):
                qc.swap(qubits[j], qubits[n - j - 1])
        for j in reversed(range(n)):
            for m in reversed(range(j + 1, n)):
                qc.cp(-np.pi / (2 ** (m - j)), qubits[m], qubits[j])
            qc.h(qubits[j])
        return

    for j in range(n):
        qc.h(qubits[j])
        for m in range(j + 1, n):
            qc.cp(np.pi / (2 ** (m - j)), qubits[m], qubits[j])
    if do_swaps:
        for j in range(n // 2):
            qc.swap(qubits[j], qubits[n - j - 1])


def apply_qft_on_valid_subspace(qc, idx, W, inverse=False):
    """
    Applies QFT only on the first W basis states of the idx register.
    Since W may not be a power of 2, this is NOT the standard QFT on 
    2^k states. Implement it as a compiled unitary:
      1. Compute the W x W DFT matrix: F[j,k] = exp(2*pi*i*j*k/W)/sqrt(W)
      2. Embed it into the 2^k x 2^k space (acting as identity on states 
         |W>, |W+1>, ..., |2^k - 1>).
      3. Convert to a UnitaryGate and append to qc on idx.
    Label the gate "QFT_W" or "iQFT_W".
    If inverse=True, use the conjugate transpose.
    """
    IDX = len(idx)
    dim = 2**IDX
    if W <= 1:
        qc.append(UnitaryGate(np.eye(dim, dtype=complex), label="iQFT_W" if inverse else "QFT_W"), list(idx))
        return

    j, k = np.meshgrid(np.arange(W), np.arange(W), indexing='ij')
    F = np.exp(2j * np.pi * j * k / W) / np.sqrt(W)
    if inverse:
        F = F.conj().T

    U = np.eye(dim, dtype=complex)
    U[:W, :W] = F
    if not np.allclose(U @ U.conj().T, np.eye(dim), atol=1e-10):
        print("Warning: embedded QFT_W is not unitary; falling back to standard QFT on 2^IDX states.")
        apply_qft(qc, idx, inverse=inverse, do_swaps=True)
        return

    qc.append(UnitaryGate(U, label="iQFT_W" if inverse else "QFT_W"), list(idx))


def apply_local_mixer(qc, idx, starts, N, mixer_angle, method="local_geometric", label=None):
    """Mezclador local que solo rota pares de indices i < W."""
    if abs(mixer_angle) < 1e-15:
        return

    IDX = len(idx)
    for a, b in mixer_edges_from_starts(starts, N, method):
        qc.append(two_level_mixer_gate(IDX, a, b, mixer_angle, label=label), list(idx))


def compute_phase_profile(costs, theta, alpha=2.0, phase_mode="cost_linear", freq_target=None):
    """Calcula phi_i de forma clasica para compilar un oraculo diagonal."""
    costs_arr = np.array(costs, dtype=float)

    if phase_mode == "cost_linear":
        phases = theta * costs_arr
        return phases.tolist(), freq_target

    if phase_mode == "cost_exponential":
        phases = theta * (np.power(alpha, costs_arr) - 1.0)
        return phases.tolist(), freq_target

    if phase_mode == "fourier_encoded":
        W = len(costs_arr)
        if W == 1:
            return [0.0], 0
        freqs = np.fft.fft(costs_arr)
        if freq_target is None:
            freqs_no_dc = freqs.copy()
            freqs_no_dc[0] = 0
            freq_target = int(np.argmax(np.abs(freqs_no_dc)))
        freq_target = int(freq_target) % W
        i = np.arange(W)
        carrier = np.exp(2j * np.pi * freq_target * i / W)
        phases = theta * np.real(freqs[freq_target] * carrier)
        return phases.tolist(), freq_target

    raise ValueError("Unknown phase_mode.")


def apply_phase_oracle(qc, idx, phi_profile, W, label="U_phi"):
    """Aplica una fase diagonal exp(i phi_i) sobre cada estado |i>."""
    for i in range(W):
        apply_phase_to_basis_state(qc, idx, i, phi_profile[i])
    qc.barrier(label=label)


In [ ]:
# =========================================================
# Construccion del circuito
# =========================================================

def build_v6_circuit(
    N, M, occupied_coords,
    theta, alpha, mixer_angle, repetitions,
    phase_mode="cost_linear",
    freq_target=None,
    use_qft=True,
    use_mixer=True,
    qft_inverse=False,
    mixer_method="local_geometric",
    add_barriers=True,
    qft_sandwich=False,
):
    validate_problem(N, M)
    D = len(N)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    order_valid = gray_order_valid(W, IDX)
    grid_bits = build_grid_bits(N, occupied_coords)
    costs = [compute_window_cost_classical(grid_bits, start, N, M) for start in starts]
    valids = [1 if c == 0 else 0 for c in costs]
    phi_profile, selected_freq = compute_phase_profile(
        costs=costs,
        theta=theta,
        alpha=alpha,
        phase_mode=phase_mode,
        freq_target=freq_target,
    )

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)
    if add_barriers:
        qc.barrier(label="init")

    for r in range(repetitions):
        apply_phase_oracle(qc, idx, phi_profile, W, label=f"U_phi[{r}]")

        if qft_sandwich and use_qft:
            apply_qft_on_valid_subspace(qc, idx, W, inverse=qft_inverse)
            if add_barriers:
                qc.barrier(label="iQFT" if qft_inverse else "QFT")

        if use_mixer:
            apply_local_mixer(qc, idx, starts, N, mixer_angle, method=mixer_method, label=f"Mix[{r}]")
            if add_barriers:
                qc.barrier(label=f"Mix[{r}]")

        if qft_sandwich and use_qft:
            apply_qft_on_valid_subspace(qc, idx, W, inverse=not qft_inverse)
            if add_barriers:
                qc.barrier(label="QFT" if qft_inverse else "iQFT")

        if add_barriers:
            qc.barrier(label=f"layer {r}")

    if (not qft_sandwich) and use_qft:
        apply_qft_on_valid_subspace(qc, idx, W, inverse=qft_inverse)
        if add_barriers:
            qc.barrier(label="iQFT final" if qft_inverse else "QFT final")

    metadata = {
        "D": D,
        "N": list(N),
        "M": list(M),
        "starts": starts,
        "W": W,
        "IDX": IDX,
        "order_valid": order_valid,
        "grid_bits": grid_bits,
        "occupied_coords": [normalize_coord(c, D) for c in occupied_coords],
        "theta": theta,
        "alpha": alpha,
        "mixer_angle": mixer_angle,
        "repetitions": repetitions,
        "phase_mode": phase_mode,
        "freq_target": selected_freq,
        "use_qft": use_qft,
        "use_mixer": use_mixer,
        "qft_inverse": qft_inverse,
        "qft_sandwich": qft_sandwich,
        "mixer_method": mixer_method,
        "phi_profile": phi_profile,
        "costs": costs,
        "valids": valids,
    }
    return qc, metadata


def index_probabilities_from_statevector(sv, metadata):
    IDX = metadata["IDX"]
    probs = np.zeros(2**IDX, dtype=float)
    idx_mask = (1 << IDX) - 1
    for basis_idx, amp in enumerate(sv.data):
        probs[basis_idx & idx_mask] += float(abs(amp) ** 2)
    return probs


def build_v4_style_baseline_circuit(
    N, M, occupied_coords, theta, mixer_angle, repetitions,
    mixer_method="local_geometric",
):
    validate_problem(N, M)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    grid_bits = build_grid_bits(N, occupied_coords)
    costs = [compute_window_cost_classical(grid_bits, start, N, M) for start in starts]
    valids = [1 if c == 0 else 0 for c in costs]
    phi_profile = (theta * np.array(costs, dtype=float)).tolist()

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)
    qc.barrier(label="init")
    for r in range(repetitions):
        apply_phase_oracle(qc, idx, phi_profile, W, label=f"U_phi[{r}]")
        apply_local_mixer(qc, idx, starts, N, mixer_angle, method=mixer_method, label=f"Mix[{r}]")
        qc.barrier(label=f"layer {r}")

    metadata = {
        "D": len(N), "N": list(N), "M": list(M), "starts": starts, "W": W, "IDX": IDX,
        "grid_bits": grid_bits, "occupied_coords": [normalize_coord(c, len(N)) for c in occupied_coords],
        "theta": theta, "alpha": None, "mixer_angle": mixer_angle, "repetitions": repetitions,
        "phase_mode": "v4_cost_linear", "freq_target": None, "use_qft": False, "use_mixer": True,
        "qft_inverse": False, "qft_sandwich": False, "mixer_method": mixer_method,
        "phi_profile": phi_profile, "costs": costs, "valids": valids,
    }
    return qc, metadata


def p_valid_from_circuit(qc, metadata):
    sv = Statevector.from_instruction(qc)
    probs = index_probabilities_from_statevector(sv, metadata)
    valid_indices = [i for i, v in enumerate(metadata["valids"]) if v == 1]
    return float(sum(probs[i] for i in valid_indices)), probs


In [ ]:
# =========================================================
# ANALYSIS 1 — Build and display main circuit
# =========================================================

qc_v6, meta_v6 = build_v6_circuit(
    N=N,
    M=M,
    occupied_coords=occupied_coords,
    theta=theta,
    alpha=alpha,
    mixer_angle=mixer_angle,
    repetitions=repetitions,
    phase_mode=phase_mode,
    freq_target=freq_target,
    use_qft=use_qft,
    use_mixer=use_mixer,
    qft_inverse=qft_inverse,
    mixer_method=mixer_method,
    qft_sandwich=qft_sandwich,
)

print("num_qubits:", qc_v6.num_qubits)
print("depth:", qc_v6.depth())
print("gate counts:", qc_v6.count_ops())

fig = qc_v6.draw("mpl")
fig.savefig("v6_circuit.png", dpi=200, bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 2 — Statevector analysis
# =========================================================

sv_v6 = Statevector.from_instruction(qc_v6)
probs_v6 = index_probabilities_from_statevector(sv_v6, meta_v6)

valid_indices_v6 = [i for i, v in enumerate(meta_v6["valids"]) if v == 1]
P_valid_before = len(valid_indices_v6) / meta_v6["W"]
P_valid_after = float(sum(probs_v6[i] for i in valid_indices_v6))
P_invalid_index_after = float(sum(probs_v6[i] for i in range(meta_v6["W"], len(probs_v6))))

print(f"N={meta_v6['N']}, M={meta_v6['M']}, W={meta_v6['W']}, IDX={meta_v6['IDX']}")
print(f"phase_mode={meta_v6['phase_mode']}, qft_sandwich={meta_v6['qft_sandwich']}, qft_inverse={meta_v6['qft_inverse']}")
print(f"theta={meta_v6['theta']:.6g}, alpha={meta_v6['alpha']:.6g}, mixer_angle={meta_v6['mixer_angle']:.6g}, repetitions={meta_v6['repetitions']}")
print()
print("index | start coord | window string | C(i) | amplitude | probability")
print("------|-------------|---------------|------|-----------|------------")

for i, start in enumerate(meta_v6["starts"]):
    window = window_string_classical(meta_v6["grid_bits"], start, meta_v6["N"], meta_v6["M"])
    amp = sv_v6.data[i] if i < len(sv_v6.data) else 0.0
    print(f"{i:5d} | {str(start):11s} | {window:13s} | {meta_v6['costs'][i]:4d} | {amp.real:+.6f}{amp.imag:+.6f}j | {probs_v6[i]:10.6f}")

print()
print(f"K={len(valid_indices_v6)}, K/W={P_valid_before:.6f}")
print(f"P_valid_before = {P_valid_before:.6f}")
print(f"P_valid_after  = {P_valid_after:.6f}")
if P_invalid_index_after > 1e-12:
    print(f"P_invalid_index_after = {P_invalid_index_after:.6f}")
if P_valid_after < P_valid_before:
    print("Warning: P_valid_after < P_valid_before; retune theta, mixer_angle, repetitions, or phase_mode.")

analysis_v6 = {
    "statevector": sv_v6,
    "probs": probs_v6,
    "valid_indices": valid_indices_v6,
    "P_valid_before": P_valid_before,
    "P_valid_after": P_valid_after,
    "P_uniform": P_valid_before,
    "P_invalid_index_after": P_invalid_index_after,
}


In [ ]:
# =========================================================
# ANALYSIS 3 — Phase profile visualization
# =========================================================

x = np.arange(meta_v6["W"])
colors = [COLOR_VALID if v == 1 else COLOR_INVALID for v in meta_v6["valids"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x, meta_v6["phi_profile"], color=colors)
axes[0].set_title("Phase profile phi_i")
axes[0].set_xlabel("index i")
axes[0].set_ylabel("phi_i")

axes[1].bar(x, meta_v6["costs"], color=colors)
axes[1].set_title("Cost profile C(i)")
axes[1].set_xlabel("index i")
axes[1].set_ylabel("C(i)")

plt.tight_layout()
fig.savefig("v6_phase_profile.png", dpi=200, bbox_inches="tight")
fig.savefig("v6_phase_profile.pdf", dpi=200, bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 4 — Amplitude distribution before vs after
# =========================================================

x = np.arange(meta_v6["W"])
before_probs = np.ones(meta_v6["W"]) / meta_v6["W"]
after_probs = analysis_v6["probs"][:meta_v6["W"]]
colors = [COLOR_VALID if v == 1 else COLOR_INVALID for v in meta_v6["valids"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.18, before_probs, width=0.36, color=colors, alpha=0.30, label="before")
ax.bar(x + 0.18, after_probs, width=0.36, color=colors, alpha=0.90, label="after")
ax.axhline(1 / meta_v6["W"], color=COLOR_BASELINE, linestyle="--", linewidth=1.5, label="1/W")
ax.set_title("Amplitude distribution before vs after")
ax.set_xlabel("index i")
ax.set_ylabel(r"probability $|\langle i|\Psi\rangle|^2$")
ax.set_xticks(x)
ax.legend()

plt.tight_layout()
fig.savefig("v6_amplitude_dist.png", dpi=200, bbox_inches="tight")
fig.savefig("v6_amplitude_dist.pdf", dpi=200, bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 5 — Phase mode comparison
# =========================================================

modes = ["cost_linear", "cost_exponential", "fourier_encoded"]
mode_scores = []
mode_metadata = {}

for mode in modes:
    qc_mode, meta_mode = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, alpha=alpha, mixer_angle=mixer_angle, repetitions=repetitions,
        phase_mode=mode, freq_target=freq_target, use_qft=use_qft, use_mixer=use_mixer,
        qft_inverse=qft_inverse, mixer_method=mixer_method, qft_sandwich=qft_sandwich,
    )
    p_valid, _ = p_valid_from_circuit(qc_mode, meta_mode)
    mode_scores.append(p_valid)
    mode_metadata[mode] = meta_mode

P_uniform = analysis_v6["P_uniform"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(modes, mode_scores, color=["#3498db", "#9b59b6", "#f39c12"], alpha=0.85)
ax.axhline(P_uniform, color=COLOR_BASELINE, linestyle="--", linewidth=1.5, label="P_uniform")
ax.set_ylim(0, max(1.0, max(mode_scores) * 1.15))
ax.set_title("Phase mode comparison")
ax.set_xlabel("phase_mode")
ax.set_ylabel("P_valid")
ax.legend()
for bar, value in zip(bars, mode_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
fig.savefig("v6_mode_comparison.png", dpi=200, bbox_inches="tight")
fig.savefig("v6_mode_comparison.pdf", dpi=200, bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 6 — QFT sandwich vs single QFT comparison
# =========================================================

rs = np.arange(1, 21)
scores_single = []
scores_sandwich = []
scores_baseline = []

for r in rs:
    qc_single, meta_single = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, alpha=alpha, mixer_angle=mixer_angle, repetitions=int(r),
        phase_mode=phase_mode, freq_target=freq_target, use_qft=True, use_mixer=use_mixer,
        qft_inverse=qft_inverse, mixer_method=mixer_method, qft_sandwich=False,
    )
    p_single, _ = p_valid_from_circuit(qc_single, meta_single)
    scores_single.append(p_single)

    qc_sandwich, meta_sandwich = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, alpha=alpha, mixer_angle=mixer_angle, repetitions=int(r),
        phase_mode=phase_mode, freq_target=freq_target, use_qft=True, use_mixer=use_mixer,
        qft_inverse=qft_inverse, mixer_method=mixer_method, qft_sandwich=True,
    )
    p_sandwich, _ = p_valid_from_circuit(qc_sandwich, meta_sandwich)
    scores_sandwich.append(p_sandwich)

    qc_base, meta_base = build_v4_style_baseline_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, mixer_angle=mixer_angle, repetitions=int(r),
        mixer_method=mixer_method,
    )
    p_base, _ = p_valid_from_circuit(qc_base, meta_base)
    scores_baseline.append(p_base)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rs, scores_single, marker="o", label="single QFT readout")
ax.plot(rs, scores_sandwich, marker="s", label="QFT sandwich")
ax.plot(rs, scores_baseline, marker="^", label="V4/V5 baseline")
ax.axhline(P_uniform, color=COLOR_BASELINE, linestyle="--", linewidth=1.5, label="P_uniform")
ax.set_title("QFT strategies vs baseline: P_valid vs repetitions")
ax.set_xlabel("repetitions")
ax.set_ylabel("P_valid")
ax.set_xticks(rs)
ax.set_ylim(0, 1.05)
ax.legend()

plt.tight_layout()
fig.savefig("v6_qft_comparison.png", dpi=200, bbox_inches="tight")
fig.savefig("v6_qft_comparison.pdf", dpi=200, bbox_inches="tight")
plt.close(fig)
